In [1]:
import os, json

DATASET_DIR = "/kaggle/input/datasets/pradhyumnakasula/abstractive-summarizer-config"  
CONFIG_PATH = f"/kaggle/input/datasets/pradhyumnakasula/abstractive-summarizer-config/project_config.json"

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"Config not found at: {CONFIG_PATH}. Check /kaggle/input for the exact folder name.")

with open(CONFIG_PATH, "r") as f:
    cfg = json.load(f)

print("✅ Loaded config:", cfg["project_name"])
print("✅ Base model:", cfg["model"]["base_model_id"])
print("✅ Dataset:", cfg["dataset"]["id"], cfg["dataset"]["config"])

print("\nFiles available in your Kaggle dataset version:")
!ls -lah "{DATASET_DIR}" | head -n 50

✅ Loaded config: abstractive_summarizer_bart_lora
✅ Base model: facebook/bart-large-cnn
✅ Dataset: abisee/cnn_dailymail 3.0.0

Files available in your Kaggle dataset version:
total 28K
drwxr-xr-x 2 nobody nogroup    0 Mar  4 18:47 .
drwxr-xr-x 3 root   root    4.0K Mar  4 18:47 ..
-rw-r--r-- 1 nobody nogroup  12K Mar  4 18:47 baseline_examples_val_200.json
-rw-r--r-- 1 nobody nogroup  134 Mar  4 18:47 baseline_rouge_val_200.json
-rw-r--r-- 1 nobody nogroup  801 Mar  4 18:47 project_config.json
-rw-r--r-- 1 nobody nogroup   54 Mar  4 18:47 run_meta.json


In [2]:
import torch, random, numpy as np

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi | head -n 20

SEED = cfg["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

CUDA: True
GPU: Tesla T4
Wed Mar  4 18:51:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------

In [3]:
from datasets import load_dataset

DATASET_ID = cfg["dataset"]["id"]
DATASET_CONFIG = cfg["dataset"]["config"]
TEXT_FIELD = cfg["dataset"]["text_field"]
SUMMARY_FIELD = cfg["dataset"]["summary_field"]

ds = load_dataset(DATASET_ID, DATASET_CONFIG)
print(ds)
print("Columns:", ds["train"].column_names)

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})
Columns: ['article', 'highlights', 'id']


In [27]:
# Started with a manageable subset but not a huge improvement on the basemodel so I'm increasing the sizes
TRAIN_N = 10000 #was 2000 initially
VAL_N = 500 #was 200 initially

train_data = ds["train"].select(range(min(TRAIN_N, len(ds["train"]))))
val_data = ds["validation"].select(range(min(VAL_N, len(ds["validation"]))))

print("Train subset:", len(train_data))
print("Val subset:", len(val_data))

Train subset: 10000
Val subset: 500


In [29]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

LORA_BASE_MODEL = "facebook/bart-large-cnn"

MAX_INPUT_TOKENS = cfg["tokenization"]["max_input_tokens"]
MAX_TARGET_TOKENS = cfg["tokenization"]["max_target_tokens"]

tokenizer = AutoTokenizer.from_pretrained(LORA_BASE_MODEL)
base_model = AutoModelForSeq2SeqLM.from_pretrained(LORA_BASE_MODEL)

device = "cuda" if torch.cuda.is_available() else "cpu"
base_model.to(device)

print("✅ LoRA base model:", LORA_BASE_MODEL)
print("Max input:", MAX_INPUT_TOKENS, "Max target:", MAX_TARGET_TOKENS)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

✅ LoRA base model: facebook/bart-large-cnn
Max input: 1024 Max target: 128


In [30]:
def preprocess(batch):
    # Inputs: article -> input_ids, attention_mask
    model_inputs = tokenizer(
        batch[TEXT_FIELD],
        max_length=MAX_INPUT_TOKENS,
        truncation=True
    )

    # Targets: highlights -> labels
    labels = tokenizer(
        text_target=batch[SUMMARY_FIELD],
        max_length=MAX_TARGET_TOKENS,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_data.map(
    preprocess,
    batched=True,
    remove_columns=train_data.column_names
)

tokenized_val = val_data.map(
    preprocess,
    batched=True,
    remove_columns=val_data.column_names
)

print(tokenized_train)
print(tokenized_val)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 10000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 500
})


In [31]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 2,359,296 || all params: 408,649,728 || trainable%: 0.5773


In [32]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/lora_runs",
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=50,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,  

    learning_rate=1e-4,
    num_train_epochs=3,             
    warmup_steps=100,

    predict_with_generate=True,
    generation_max_length=MAX_TARGET_TOKENS,

    fp16=True,  
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

print("✅ Trainer initialized.")

✅ Trainer initialized.


In [33]:
!pip install -q evaluate rouge_score

In [37]:
import numpy as np
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    # Some versions return (preds, ...) as a tuple
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100 in labels (ignore index) before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    scores = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    return {k: float(v) for k, v in scores.items()}

trainer.compute_metrics = compute_metrics
print("✅ compute_metrics attached (ROUGE).")

✅ compute_metrics attached (ROUGE).


In [38]:
train_result = trainer.train() #took around 3 hours to run on a GPU Phew!!! so be careful whenever you run it tips:- you can cut down the epochs, increase learning rate, reduce train_n, val_n
train_result

Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
200,38.717300,1.708257,0.346833,0.150329,0.249930,0.309202
400,37.640032,1.700183,0.348937,0.150719,0.251018,0.312383


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=471, training_loss=43.3718391305069, metrics={'train_runtime': 9183.9366, 'train_samples_per_second': 3.267, 'train_steps_per_second': 0.051, 'total_flos': 6.249633263163802e+16, 'train_loss': 43.3718391305069, 'epoch': 3.0})

In [39]:
metrics = trainer.evaluate()
metrics

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'eval_loss': 1.703834891319275,
 'eval_rouge1': 0.3511547677770688,
 'eval_rouge2': 0.15307192595687213,
 'eval_rougeL': 0.25429194621114004,
 'eval_rougeLsum': 0.3151387165208254,
 'eval_runtime': 717.7582,
 'eval_samples_per_second': 0.697,
 'eval_steps_per_second': 0.174,
 'epoch': 3.0}

In [40]:
import json

METRICS_PATH = "/kaggle/working/lora_eval_metrics_val_500.json"

metrics_out = {k: float(v) for k, v in metrics.items()}  # convert np/torch types to plain floats

with open(METRICS_PATH, "w") as f:
    json.dump(metrics_out, f, indent=2)

print("✅ Saved metrics to:", METRICS_PATH)

✅ Saved metrics to: /kaggle/working/lora_eval_metrics_val_500.json


In [41]:
import torch, json

device = "cuda" if torch.cuda.is_available() else "cpu"
EXAMPLES_PATH = "/kaggle/working/lora_examples_val_15.json"
N_EX = 15

def summarize_one(text: str):
    inputs = tokenizer(
        text,
        max_length=MAX_INPUT_TOKENS,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_TOKENS,
            num_beams=4,
            length_penalty=2.0,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return tokenizer.decode(out_ids[0], skip_special_tokens=True)

examples = []
for i in range(N_EX):
    article = val_data[i][TEXT_FIELD]
    ref = val_data[i][SUMMARY_FIELD]
    pred = summarize_one(article)

    examples.append({
        "i": i,
        "article_preview": article[:800],
        "reference": ref,
        "lora_prediction": pred
    })

with open(EXAMPLES_PATH, "w") as f:
    json.dump(examples, f, indent=2)

print("✅ Saved examples to:", EXAMPLES_PATH)

✅ Saved examples to: /kaggle/working/lora_examples_val_15.json


## 03 — LoRA Fine-Tuning of BART for Summarization

In this notebook I fine-tune a pretrained summarization model using **LoRA (Low-Rank Adaptation)** on the **CNN/DailyMail dataset**.

The goal of this stage is to adapt the pretrained model to the dataset while training only a small number of additional parameters instead of updating the full model.

---

### Model

The base model used is: facebook/bart-large-cnn
This model is already pretrained for abstractive summarization and serves as a strong starting point.

---

### Parameter-Efficient Fine-Tuning (LoRA)

Instead of full fine-tuning, I use **LoRA adapters** applied to the attention layers of the model.

Configuration used:
r = 16
lora_alpha = 32
lora_dropout = 0.05
This allows the model to learn task-specific behavior while keeping the majority of the pretrained parameters frozen.

---

### Dataset

The **CNN/DailyMail dataset** is used for training.

For faster experimentation I trained on a subset:
Training samples: 10,000
Validation samples: 500
Each example consists of:

- `article` → full news article  
- `highlights` → reference summary

---

### Training Setup

The model is trained using HuggingFace's `Seq2SeqTrainer`.

Key training parameters:
learning_rate = 1e-4
epochs = 3
batch_size = 2
gradient_accumulation = 8

Mixed precision (`fp16`) is used to efficiently utilize the GPU.

---

### Evaluation

Model performance is evaluated using **ROUGE**, a standard summarization metric measuring overlap between generated summaries and reference summaries.

Final validation results:
ROUGE-1 ≈ 0.351
ROUGE-2 ≈ 0.153
ROUGE-L ≈ 0.254
ROUGE-Lsum ≈ 0.315
### Outputs Saved

The following artifacts were generated and saved:

- `lora_eval_metrics_val_500.json`  
  → ROUGE evaluation results

- `lora_examples_val_15.json`  
  → sample generated summaries for qualitative analysis

- `lora_adapter_bart_large_cnn.zip`  
  → trained LoRA adapter weights and tokenizer files

These outputs are used in the next notebook to compare **baseline vs LoRA performance**.